In [1]:
import geopandas as gpd
import  os 
import pandas as pd 
usuario = os.getlogin()


In [2]:

# yucatan = gpd.read_file(fr"C:\Users\{usuario}\Downloads\Anexo_1_yuc_validado.gpkg")

In [3]:
base = gpd.read_file(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\UPN_IMO_IMB_CASASSALUD_sin_CESSA.gpkg")

In [4]:
sus = gpd.read_file(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\UPN_IMO_IMB_CASASSALUD_sin_CESSA.gpkg")

In [5]:
consultorio = pd.read_excel(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\asignacion_definitiva_y_aprovechamiento.xlsx",sheet_name="Unidades",skiprows=2)

In [6]:
productividad =  pd.read_parquet(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\productividad_coplamar_imb.parquet")

In [7]:
unid = gpd.read_file(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\Unidades aceptadas_completo3.gpkg")
poligono = gpd.read_file(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\Disolución de polígonos por CLUES.gpkg")

In [8]:
# poligono = poligono.rename(columns={"CLUES":"CLUES_1"})

In [9]:
poligono.columns = poligono.columns.str.lower().str.replace(' ', '_')

In [10]:
unid.columns = unid.columns.str.lower().str.replace(' ', '_')

In [11]:
consultorio.columns = consultorio.columns.str.lower().str.replace(' ', '_')

In [12]:
poligono.columns

Index(['fid', 'cvegeo', 'pobtot', 'pobfem', 'pobmas', 'p_0a2', 'pob0_14',
       'p_60ymas', 'p_60ymas_f', 'p_60ymas_m', 'p_15a49_f', 'psinder',
       'pder_ss', 'pder_imss', 'pder_segp', 'pder_imssb', 'pder_iste',
       'pder_istee', 'pafil_pdom', 'sin_der', 'nom_mun', 'dist_km', 'dur_min',
       'clues', 'institucion', 'nombre_unidad',
       'población_con_derechohabiencia_2026',
       'población_sin_derechohabiencia_2026', 'población_total_2026',
       'geometry'],
      dtype='object')

In [13]:
diagnostico_geometrico = {
    "poligono_crs": str(poligono.crs),
    "poligono_tipos": poligono.geom_type.value_counts().to_dict(),
    "poligono_registros": len(poligono),
    "poligono_clues_unicas": poligono["clues"].nunique(dropna=True),
    "poligono_geometrias_validas": int(poligono.geometry.is_valid.sum()),
    "unid_crs": str(unid.crs),
    "unid_tipos": unid.geom_type.value_counts().to_dict(),
    "unid_registros": len(unid),
    "unid_clues_unicas": unid["clues"].nunique(dropna=True),
}
diagnostico_geometrico

{'poligono_crs': 'EPSG:4326',
 'poligono_tipos': {'MultiPolygon': 11601},
 'poligono_registros': 11601,
 'poligono_clues_unicas': 11600,
 'poligono_geometrias_validas': 11601,
 'unid_crs': 'EPSG:4326',
 'unid_tipos': {'Point': 8600},
 'unid_registros': 8600,
 'unid_clues_unicas': 5041}

In [14]:
poligono_metrico = poligono.to_crs("EPSG:6372")
complejidad_original = int(poligono_metrico.geometry.get_coordinates().shape[0])
pruebas_simplificacion = {}

for tolerancia_m in (100, 250, 500):
    geometria_simplificada = poligono_metrico.geometry.simplify(
        tolerancia_m,
        preserve_topology=True,
    )
    pruebas_simplificacion[tolerancia_m] = {
        "coordenadas": int(geometria_simplificada.get_coordinates().shape[0]),
        "reduccion_pct": round(
            (1 - geometria_simplificada.get_coordinates().shape[0] / complejidad_original) * 100,
            1,
        ),
    }

{"coordenadas_originales": complejidad_original, "pruebas": pruebas_simplificacion}

{'coordenadas_originales': 880158,
 'pruebas': {100: {'coordenadas': 285629, 'reduccion_pct': 67.5},
  250: {'coordenadas': 212803, 'reduccion_pct': 75.8},
  500: {'coordenadas': 156644, 'reduccion_pct': 82.2}}}

In [15]:
import json
import zlib
from pathlib import Path

salida_voronoi = Path.cwd() / "public" / "voronoi"
salida_voronoi.mkdir(parents=True, exist_ok=True)

for archivo_anterior in salida_voronoi.glob("*.geojson"):
    archivo_anterior.unlink()

voronoi_web = poligono.loc[
    poligono["clues"].notna(),
    ["clues", "geometry"],
].copy()
voronoi_web["clues"] = voronoi_web["clues"].astype(str).str.strip()
voronoi_web["fragmento_id"] = voronoi_web["clues"].map(
    lambda clues: zlib.crc32(clues.encode("utf-8")) % 32
)
voronoi_web = voronoi_web.to_crs("EPSG:6372")
voronoi_web["geometry"] = voronoi_web.geometry.simplify(100, preserve_topology=True)
voronoi_web = voronoi_web.to_crs("EPSG:4326")

indice_voronoi = {}
for fragmento_id, fragmento in voronoi_web.groupby("fragmento_id", sort=True):
    nombre_archivo = f"fragmento-{fragmento_id:02d}.geojson"
    fragmento_salida = fragmento[["clues", "geometry"]].copy()
    (salida_voronoi / nombre_archivo).write_text(
        fragmento_salida.to_json(drop_id=True, separators=(",", ":")),
        encoding="utf-8",
    )
    indice_voronoi.update(dict.fromkeys(fragmento_salida["clues"], nombre_archivo))

(salida_voronoi / "index.json").write_text(
    json.dumps(indice_voronoi, ensure_ascii=False, separators=(",", ":")),
    encoding="utf-8",
)

tamanos_fragmentos = [archivo.stat().st_size for archivo in salida_voronoi.glob("*.geojson")]
tamano_mb = sum(archivo.stat().st_size for archivo in salida_voronoi.iterdir()) / 1024**2
print(f"Voronoi exportado: {len(voronoi_web):,} polígonos en {len(tamanos_fragmentos)} fragmentos")
print(f"Índice: {len(indice_voronoi):,} CLUES")
print(f"Tamaño total: {tamano_mb:.1f} MB")
print(f"Fragmento mayor: {max(tamanos_fragmentos) / 1024**2:.1f} MB")

Voronoi exportado: 11,600 polígonos en 32 fragmentos
Índice: 11,600 CLUES
Tamaño total: 12.2 MB
Fragmento mayor: 0.4 MB


In [16]:
for nombre, gdf in {"poligono": poligono, "unid": unid}.items():
    print(nombre)
    print("CRS:", gdf.crs)
    print(gdf.geom_type.value_counts())
    print("Registros:", len(gdf))

poligono
CRS: EPSG:4326
MultiPolygon    11601
Name: count, dtype: int64
Registros: 11601
unid
CRS: EPSG:4326
Point    8600
Name: count, dtype: int64
Registros: 8600


In [17]:
unid.columns

Index(['uid', 'id_temp_sus', 'id_temp_sheets_edos', 'operador',
       'institucion_que_propone_espacio', 'entidad', 'municipio', 'nombre',
       'clues', 'estrato', 'mod_id', 'modalidad', 'lat', 'lon',
       'origen_estrato', 'cluster_rural_id', 'cluster_rural_tamano_unidades',
       'regla_cluster', 'motivo_decision', 'estatus_final', 'num_consultorios',
       'fid', 'cvegeo', 'pobtot', 'pobfem', 'pobmas', 'p_0a2', 'pob0_14',
       'p_60ymas', 'p_60ymas_f', 'p_60ymas_m', 'p_15a49_f', 'psinder',
       'pder_ss', 'pder_imss', 'pder_segp', 'pder_imssb', 'pder_iste',
       'pder_istee', 'pafil_pdom', 'sin_der', 'nom_mun', 'dist_km', 'dur_min',
       'clues_2', 'institucion', 'nombre_unidad',
       'población_con_derechohabiencia_2026',
       'población_sin_derechohabiencia_2026', 'población_total_2026',
       'geometry'],
      dtype='object')

In [18]:
productividad

,clues,nombre_de_la_unidad,entidad,cve_ent,municipio,cve_mun,localidad,cve_loc,cvegeo,poblacion_con_derechohabiencia_2026,consulta_general
0,YNSSA014046,CENTRO DE SALUD DE MOTUL,YUCATAN,31,MOTUL,052,MOTUL DE CARRILLO PUERTO,0001,310520001,0.0,NaN
1,HGIMB000035,SAN BARTOLO,HIDALGO,13,ACATLAN,001,SAN BARTOLO (EL LLANO),0020,130010020,0.0,2251.0
2,SPIMB002516,CENTRO DE SALUD HUEHUETLAN,SAN LUIS POTOSI,24,HUEHUETLAN,018,HUEHUETLAN,0001,240180001,1.0,5824.0
3,YNSSA000705,CENTRO DE SALUD XOHUAYAN,YUCATAN,31,OXKUTZCAB,056,XOHUAYAN,0013,310560013,10.0,NaN
4,CMIMB000260,CENTRO DE SALUD CAMPO CUATRO,COLIMA,06,COMALA,003,CAMPO CUATRO,0008,060030008,11.0,807.0
...,...,...,...,...,...,...,...,...,...,...,...
11595,SRIMB002401,CENTRO DE SALUD URBANO NOGALES,SONORA,26,NOGALES,043,HEROICA NOGALES,0001,260430001,179373.0,17753.0
11596,YNSSA000466,CENTRO DE SALUD KANASÍN,YUCATAN,31,KANASIN,041,KANASIN,0001,310410001,190863.0,NaN
11597,QRIMB001430,CENTRO DE SALUD URBANO 01 PLAYA DEL CARMEN,QUINTANA ROO,23,SOLIDARIDAD,008,PLAYA DEL CARMEN,0001,230080001,182283.0,1900.0
11598,SLIMB003012,CENTRO DE SALUD MOCHIS II,SINALOA,25,AHOME,001,LOS MOCHIS,0001,250010001,193982.0,59650.0


In [19]:
productividad

,clues,nombre_de_la_unidad,entidad,cve_ent,municipio,cve_mun,localidad,cve_loc,cvegeo,poblacion_con_derechohabiencia_2026,consulta_general
0,YNSSA014046,CENTRO DE SALUD DE MOTUL,YUCATAN,31,MOTUL,052,MOTUL DE CARRILLO PUERTO,0001,310520001,0.0,NaN
1,HGIMB000035,SAN BARTOLO,HIDALGO,13,ACATLAN,001,SAN BARTOLO (EL LLANO),0020,130010020,0.0,2251.0
2,SPIMB002516,CENTRO DE SALUD HUEHUETLAN,SAN LUIS POTOSI,24,HUEHUETLAN,018,HUEHUETLAN,0001,240180001,1.0,5824.0
3,YNSSA000705,CENTRO DE SALUD XOHUAYAN,YUCATAN,31,OXKUTZCAB,056,XOHUAYAN,0013,310560013,10.0,NaN
4,CMIMB000260,CENTRO DE SALUD CAMPO CUATRO,COLIMA,06,COMALA,003,CAMPO CUATRO,0008,060030008,11.0,807.0
...,...,...,...,...,...,...,...,...,...,...,...
11595,SRIMB002401,CENTRO DE SALUD URBANO NOGALES,SONORA,26,NOGALES,043,HEROICA NOGALES,0001,260430001,179373.0,17753.0
11596,YNSSA000466,CENTRO DE SALUD KANASÍN,YUCATAN,31,KANASIN,041,KANASIN,0001,310410001,190863.0,NaN
11597,QRIMB001430,CENTRO DE SALUD URBANO 01 PLAYA DEL CARMEN,QUINTANA ROO,23,SOLIDARIDAD,008,PLAYA DEL CARMEN,0001,230080001,182283.0,1900.0
11598,SLIMB003012,CENTRO DE SALUD MOCHIS II,SINALOA,25,AHOME,001,LOS MOCHIS,0001,250010001,193982.0,59650.0


In [20]:
consultorio.columns


Index(['escenario', 'clues', 'tipo_de_unidad', 'localidades',
       'población_total_2026', 'población_sin_derechohabiencia_2026',
       'consultorios_habilitados', 'consultorios_inhabilitados',
       'total_consultorios', 'consultorios_usados',
       'población_por_consultorio', 'razón_de_aprovechamiento', 'categoría',
       'supuesto_1_consultorio'],
      dtype='object')

In [21]:
len(consultorio)

30344

In [22]:
len(base)

15323

In [23]:
base.columns

Index(['clues', 'nombre_unidad', 'institucion', 'entidad', 'municipio',
       'geometry'],
      dtype='object')

In [24]:
base = base.merge(
    consultorio[['clues', 'total_consultorios', 'población_por_consultorio']],
    on='clues',
    how='left'
)

In [25]:
base = base.merge(
    productividad[['clues','consulta_general']],
    on='clues',
    how='left'
)

In [26]:
len(base)

29917

In [27]:
base.columns

Index(['clues', 'nombre_unidad', 'institucion', 'entidad', 'municipio',
       'geometry', 'total_consultorios', 'población_por_consultorio',
       'consulta_general'],
      dtype='object')

In [28]:
# base = base[base["nivel_atencion"]=="PRIMER NIVEL"]

In [29]:
# tipologias = [
#     'CENTROS DE SALUD CON SERVICIOS AMPLIADOS',
#     'UNIDAD MÓVIL',
#     'URBANO DE 01 NÚCLEOS BÁSICOS',
#     'URBANO DE 05 NÚCLEOS BÁSICOS',
#     'RURAL DE 01 NÚCLEO BÁSICO',
#     'URBANO DE 02 NÚCLEOS BÁSICOS',
#     'URBANO DE 11 NÚCLEOS BÁSICOS',
#     'URBANO DE 12 NÚCLEOS BÁSICOS Y MÁS',
#     'RURAL DE 02 NÚCLEOS BÁSICOS',
#     'URBANO DE 06 NÚCLEOS BÁSICOS',
#     'URBANO DE 03 NÚCLEOS BÁSICOS',
#     'URBANO DE 07 NÚCLEOS BÁSICOS',
#     'URBANO DE 08 NÚCLEOS BÁSICOS',
#     'URBANO DE 04 NÚCLEOS BÁSICOS'
# ]

# base = base[base['NOMBRE DE TIPOLOGIA'].isin(tipologias)]

In [30]:
conteo_imo = base["institucion"].unique()
conteo_imo

array(['IMB', 'IMS', 'CSA'], dtype=object)

In [31]:
base.columns

Index(['clues', 'nombre_unidad', 'institucion', 'entidad', 'municipio',
       'geometry', 'total_consultorios', 'población_por_consultorio',
       'consulta_general'],
      dtype='object')

In [32]:
len(base)

29917

In [33]:
from pathlib import Path

columnas_mapa = [
    'clues', 'nombre_unidad', 'institucion', 'entidad', 'municipio', 'geometry'
]

consultorio_mapa = consultorio.loc[:, [
    'clues', 'escenario', 'total_consultorios', 'población_por_consultorio'
]].copy()
consultorio_mapa['clues'] = consultorio_mapa['clues'].astype(str).str.strip()
consultorio_mapa['_prioridad'] = (
    consultorio_mapa['escenario'].astype('string').str.strip().str.casefold()
    != 'Después'.casefold()
)
consultorio_mapa = (
    consultorio_mapa
    .sort_values('_prioridad', kind='stable')
    .drop_duplicates(subset='clues', keep='first')
    .drop(columns=['escenario', '_prioridad'])
)

productividad_mapa = productividad.loc[:, ['clues', 'consulta_general']].copy()
productividad_mapa['clues'] = productividad_mapa['clues'].astype(str).str.strip()

base_mapa = base.loc[
    base["institucion"].isin(["IMS", "IMB", "CSA"]),
    columnas_mapa,
].copy()

base_mapa["institucion"] = base_mapa["institucion"].replace({"IMS": "IMO"})
base_mapa['clues'] = base_mapa['clues'].astype(str).str.strip()
base_mapa = base_mapa.merge(consultorio_mapa, on='clues', how='left', validate='many_to_one')
base_mapa = base_mapa.merge(productividad_mapa, on='clues', how='left', validate='many_to_one')

salida_mapa = Path.cwd() / "public" / "mapa_base.geojson"
base_mapa.to_file(salida_mapa, driver="GeoJSON")

print(base_mapa["institucion"].value_counts().to_dict())
print(f"GeoJSON generado: {salida_mapa} ({len(base_mapa):,} CLUES)")
print(f"CLUES con consultorios: {base_mapa['total_consultorios'].notna().sum():,}")
print(f"CLUES con consulta general: {base_mapa['consulta_general'].notna().sum():,}")

{'IMB': 16075, 'CSA': 7398, 'IMO': 6444}
GeoJSON generado: c:\Users\jose.valdez\Downloads\mapa_imo\mapa_primer_nivel\public\mapa_base.geojson (29,917 CLUES)
CLUES con consultorios: 16,048
CLUES con consulta general: 21,956
